# 🔍 02 - AI-Powered Action Extraction

**COMPLETELY INDEPENDENT NOTEBOOK** - Run this after notebook 01.

## What this notebook does:
- ✅ Loads data from `quickstart_catalog_vkm_external.classify_tickets.raw_tickets`  
- ✅ Uses **Databricks AI Functions** for intelligent extraction
- ✅ Extracts actions, priorities, and timelines using LLMs
- ✅ Categorizes action items by type  
- ✅ Saves results to Unity Catalog tables  
- ✅ Ready for next notebook: `03_ai_classification.ipynb`

**Prerequisites:** Run `01_sample_data_generation.ipynb` first

## AI Functions Used:
- `ai_classify` - For priority and category classification
- `ai_extract` - For action items extraction  
- `ai_gen` - For complex analysis and summaries


In [1]:
# Import required libraries
import pandas as pd
import re
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

# Import configuration
%run ./config

# Define a UDF to extract JSON from AI responses
def extract_json_from_ai_response(text):
    """Extract JSON array from AI response text, handling various formats"""
    if not text:
        return "[]"
    
    # Try to find JSON array pattern
    # Look for [ followed by content and ending with ]
    pattern = r'\[(.*?)\]'
    matches = re.findall(pattern, text, re.DOTALL)
    
    if matches:
        # Take the first match and reconstruct the JSON array
        content = matches[0].strip()
        # Clean up the content - remove extra whitespace and newlines
        content = re.sub(r'\s+', ' ', content)
        return f"[{content}]"
    
    # If no array found, try to extract quoted strings
    quoted_items = re.findall(r'"([^"]+)"', text)
    if quoted_items:
        json_items = [f'"{item}"' for item in quoted_items]
        return f"[{', '.join(json_items)}]"
    
    return "[]"

# Register the UDF
extract_json_udf = udf(extract_json_from_ai_response, StringType())

print("✅ Libraries imported, configuration loaded, and JSON extraction UDF defined")


✅ Libraries imported, configuration loaded, and JSON extraction UDF defined


In [2]:
# Load Sample Data
print("📊 Loading sample ticket data from Unity Catalog...")

# Load the sample ticket data from Unity Catalog
df_tickets = spark.table(TABLES["raw_tickets"])
print(f"✅ Loaded {df_tickets.count()} tickets from: {TABLES['raw_tickets']}")

# Display sample data
print("\n📋 Sample data:")
display(df_tickets.select("ticket_id", "short_description", "description"))


📊 Loading sample ticket data from Unity Catalog...


✅ Loaded 10 tickets from: quickstart_catalog_vkm_external.classify_tickets.raw_tickets

📋 Sample data:


,ticket_id,short_description,description
0,TICKET_001,Issue #1 - Critical,hey so our website is super slow today and customers are complaining. i think it might be the database or something. can someone look into this? it's been happening since this morning and we're losing sales.
1,TICKET_002,Issue #2 - Important,urgent! the login system is broken again. users can't get in and they're calling support nonstop. this happened last week too. we need to fix this asap before more customers leave.
2,TICKET_003,Issue #3 - Help needed,i need help with the new feature we're building. the api is returning weird errors and i don't know why. it works sometimes but then fails randomly. can someone debug this?
3,TICKET_004,Issue #4 - Important,our monitoring alerts are going crazy. everything shows red but the app seems to be working fine. not sure if it's a false positive or if something is actually broken. help?
4,TICKET_005,Issue #5 - Urgent,the deployment failed again and now production is down. we rolled back but need to figure out what went wrong. this is the third time this month. we need better testing.
5,TICKET_006,Issue #6 - Problem,customers are reporting that their data is missing after the last update. this is a big problem and we need to investigate immediately. could be a data migration issue.
6,TICKET_007,Issue #7 - Critical,the new security patch broke our authentication. users can't log in and we're getting flooded with support tickets. need to fix this before the security team gets involved.
7,TICKET_008,Issue #8 - Important,our cloud costs are through the roof this month. something is using way more resources than usual. need to find out what's causing the spike and optimize it.
8,TICKET_009,Issue #9 - Urgent,the backup system isn't working and we haven't had a successful backup in 3 days. this is a major risk and we need to fix it before something bad happens.
9,TICKET_010,Issue #10 - Critical,the new microservice is causing memory leaks and crashing the whole system. we need to either fix it or disable it until we can figure out what's wrong.


In [3]:
# AI-Powered Extraction Using Databricks AI Functions
print("🤖 Step 1: Classifying ticket priorities using ai_classify...")

# 1. PRIORITY CLASSIFICATION using ai_classify (SQL syntax)
try:
    # Register DataFrame as temporary view for SQL access
    df_tickets.createOrReplaceTempView("tickets")
    
    # Use SQL to call ai_classify
    df_with_priority = spark.sql("""
        SELECT
            *,
            ai_classify(description, ARRAY('Low Priority', 'Medium Priority', 'High Priority', 'Urgent Priority')) as priority_classified
        FROM tickets
    """)
    print("✅ Priority classification completed")
except Exception as e:
    print(f"❌ Error in priority classification: {e}")
    print("🔧 Using rule-based fallback approach...")
    
    # Rule-based priority classification
    df_with_priority = df_tickets.withColumn(
        "priority_classified",
        when(col("description").rlike("(?i)(urgent|critical|emergency|asap|immediately|now)"), lit("Urgent Priority"))
        .when(col("description").rlike("(?i)(high|important|priority|soon|today)"), lit("High Priority"))
        .when(col("description").rlike("(?i)(low|minor|sometime|later|when possible)"), lit("Low Priority"))
        .otherwise(lit("Medium Priority"))
    )
    print("✅ Rule-based priority classification completed")

display(df_with_priority.select("ticket_id", "short_description", "priority_classified"))


🤖 Step 1: Classifying ticket priorities using ai_classify...
✅ Priority classification completed


,ticket_id,short_description,priority_classified
0,TICKET_001,Issue #1 - Critical,Urgent Priority
1,TICKET_002,Issue #2 - Important,Urgent Priority
2,TICKET_003,Issue #3 - Help needed,Medium Priority
3,TICKET_004,Issue #4 - Important,High Priority
4,TICKET_005,Issue #5 - Urgent,High Priority
5,TICKET_006,Issue #6 - Problem,Urgent Priority
6,TICKET_007,Issue #7 - Critical,Urgent Priority
7,TICKET_008,Issue #8 - Important,High Priority
8,TICKET_009,Issue #9 - Urgent,Urgent Priority
9,TICKET_010,Issue #10 - Critical,Urgent Priority


In [ ]:
# 2. ACTION ITEMS EXTRACTION using ai_gen (better for arrays)
print("🔍 Step 2: Extracting action items using ai_gen...")

# Check if AI functions are available by testing with a simple query
ai_functions_available = False
try:
    # Test AI function availability with a very simple query
    test_df = spark.sql("SELECT ai_gen('test') as test_output")
    test_df.collect()  # Force execution
    ai_functions_available = True
    print("✅ AI functions are available")
except Exception as e:
    print(f"❌ AI functions not available: {e}")
    print("🔧 Using fallback approach without AI functions...")
    ai_functions_available = False

if ai_functions_available:
    try:
        # Register DataFrame as temporary view for SQL access
        df_with_priority.createOrReplaceTempView("tickets_with_priority")
        
        # Use SQL to call ai_gen
        df_with_actions = spark.sql("""
            SELECT
                *,
                ai_gen(
                    CONCAT(
                        'Extract all specific action items, tasks, requirements, and deliverables from this ticket description. Return as a JSON array of strings. Each item should be a concrete, actionable task. If no action items are found, return an empty array []. Ticket description: ',
                        description
                    )
                ) AS action_items_raw
            FROM tickets_with_priority
        """)
        print("✅ AI-powered action extraction completed")
        
    except Exception as e:
        print(f"❌ Error in AI extraction: {e}")
        print("🔧 Falling back to non-AI approach...")
        ai_functions_available = False

if not ai_functions_available:
    # Fallback: Create a simple extraction without AI
    print("🔧 Using rule-based extraction as fallback...")
    
    # Simple rule-based extraction - look for common action words
    df_with_actions = df_with_priority.withColumn(
        "action_items_raw",
        when(
            col("description").rlike("(?i)(install|setup|configure|deploy|update|fix|resolve|implement|create|build|test|monitor|backup|restore|migrate|upgrade|patch|secure|optimize|troubleshoot|investigate|analyze|review|document|train|schedule|plan|design|develop|maintain|manage|administer|operate|support|service)"),
            lit('["Action item extracted from description"]')
        ).otherwise(lit("[]"))
    )
    print("✅ Rule-based action extraction completed")

# Debug: Let's see what the raw output looks like
print("🔍 Debug: Raw AI output for first ticket:")
df_with_actions.select("ticket_id", "action_items_raw").show(1, truncate=False)

# Clean the raw AI output to extract pure JSON using UDF
# The AI often returns JSON wrapped in markdown code blocks, so we need to extract just the JSON part
df_with_actions_cleaned = df_with_actions.withColumn(
    "action_items_cleaned",
    extract_json_udf(col("action_items_raw"))
)

# Debug: Let's see what the cleaned output looks like
print("🔍 Debug: Cleaned JSON for first ticket:")
df_with_actions_cleaned.select("ticket_id", "action_items_cleaned").show(1, truncate=False)

# Additional debug: Let's see the full raw output for debugging
print("🔍 Debug: Full raw output for first ticket:")
df_with_actions.select("ticket_id", "action_items_raw").show(1, truncate=False)

# Parse the cleaned JSON response and convert to array
try:
    df_with_actions_parsed = df_with_actions_cleaned.withColumn(
        "action_items_extracted",
        from_json(col("action_items_cleaned"), ArrayType(StringType()))
    ).withColumn(
        "action_items_extracted",
        when(col("action_items_extracted").isNull(), array()).otherwise(col("action_items_extracted"))
    ).drop("action_items_raw", "action_items_cleaned")
    print("✅ JSON parsing completed successfully")
except Exception as e:
    print(f"❌ Error in JSON parsing: {e}")
    print("🔧 Using fallback parsing approach...")
    
    # Fallback: Create action_items_extracted directly from the raw data
    df_with_actions_parsed = df_with_actions_cleaned.withColumn(
        "action_items_extracted",
        when(col("action_items_cleaned") == "[]", array())
        .when(col("action_items_cleaned").rlike(r'^\[.*\]$'), 
              split(regexp_replace(col("action_items_cleaned"), r'[\[\]"]', ''), ','))
        .otherwise(array(col("action_items_cleaned")))
    ).drop("action_items_raw", "action_items_cleaned")
    print("✅ Fallback parsing completed")

print("✅ Action items extraction completed")

# Debug: Let's see the final parsed action items
print("🔍 Debug: Final parsed action items for first ticket:")
try:
    df_with_actions_parsed.select("ticket_id", "action_items_extracted").show(1, truncate=False)
    
    # Additional debugging - check the schema and data types
    print("🔍 Debug: Schema of action_items_extracted column:")
    df_with_actions_parsed.select("action_items_extracted").dtypes
    
    print("🔍 Debug: Sample of raw action_items_raw:")
    df_with_actions_parsed.select("ticket_id", "action_items_raw").show(2, truncate=False)
    
except Exception as e:
    print(f"❌ Error in debug section: {e}")
    print("🔍 Debug: Available columns:")
    print(df_with_actions_parsed.columns)

# Additional debug: Check if any tickets have non-empty action items
print("🔍 Debug: Count of tickets with non-empty action items:")
try:
    # Check if action_items_extracted column exists and is an array
    if "action_items_extracted" in df_with_actions_parsed.columns:
        # Try to count non-empty arrays
        non_empty_count = df_with_actions_parsed.filter(
            col("action_items_extracted").isNotNull() & 
            (size(col("action_items_extracted")) > 0)
        ).count()
        print(f"   Non-empty action items: {non_empty_count}")
        
        # Show sample of tickets with action items
        print("🔍 Debug: Sample of tickets with action items:")
        df_with_actions_parsed.filter(
            col("action_items_extracted").isNotNull() & 
            (size(col("action_items_extracted")) > 0)
        ).select("ticket_id", "action_items_extracted").show(5, truncate=False)
    else:
        print("   action_items_extracted column not found")
except Exception as e:
    print(f"   Error checking action items: {e}")
    print("   Showing first few rows of action_items_extracted column:")
    df_with_actions_parsed.select("ticket_id", "action_items_extracted").show(5, truncate=False)

display(df_with_actions_parsed.select("ticket_id", "short_description", "action_items_extracted"))


🔍 Step 2: Extracting action items using ai_gen...
✅ AI functions are available
🔍 Debug: Raw AI output for first ticket:


+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ticket_id |action_items_cleaned                                                                                                                                            |
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|TICKET_001|["Investigate the cause of the website's slow performance", "Check the database for potential issues", "Look into the website's performance since this morning"]|
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
only showing top 1 row
🔍 Debug: Full raw output for first ticket:


+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ticket_id |action_items_raw                                                                                                                                                                                                                                                                                                                                                                                                                                                          

+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|ticket_id |action_items_extracted                                                                                                                                    |
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|TICKET_001|[Investigate the cause of the website's slow performance, Check the database for potential issues, Look into the website's performance since this morning]|
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
only showing top 1 row
🔍 Debug: Count of tickets with non-empty action items:


🔍 Debug: Sample of tickets with action items:


+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|ticket_id |action_items_extracted                                                                                                                                    |
+----------+----------------------------------------------------------------------------------------------------------------------------------------------------------+
|TICKET_001|[Investigate the cause of the website's slow performance, Check the database for potential issues, Look into the website's performance since this morning]|
|TICKET_002|[Fix the login system, Investigate the cause of the login system failure, Prevent the login system from breaking again]                                   |
|TICKET_003|[Debug the API to identify the cause of the random errors]                                                                                          

,ticket_id,short_description,action_items_extracted
0,TICKET_001,Issue #1 - Critical,"[Investigate the cause of the website's slow performance, Check the database for potential issues, Look into the website's performance since this morning]"
1,TICKET_002,Issue #2 - Important,"[Fix the login system, Investigate the cause of the login system failure, Prevent the login system from breaking again]"
2,TICKET_003,Issue #3 - Help needed,[Debug the API to identify the cause of the random errors]
3,TICKET_004,Issue #4 - Important,"[Investigate monitoring alerts, Determine cause of false positives or actual issues, Verify app functionality]"
4,TICKET_005,Issue #5 - Urgent,"[Figure out what went wrong with the deployment, Improve testing to prevent similar failures in the future]"
5,TICKET_006,Issue #6 - Problem,"[Investigate the cause of missing customer data, Check for data migration issues]"
6,TICKET_007,Issue #7 - Critical,"[Fix the authentication issue caused by the new security patch, Resolve the login problem for users, Reduce the influx of support tickets related to authentication]"
7,TICKET_008,Issue #8 - Important,"[Investigate the cause of the cloud cost spike, Identify the resource(s) using more resources than usual, Optimize the resource(s) causing the spike]"
8,TICKET_009,Issue #9 - Urgent,"[Investigate the backup system to determine the cause of the failure, Fix the backup system to enable successful backups, Verify a successful backup has occurred to ensure the system is working correctly]"
9,TICKET_010,Issue #10 - Critical,"[Fix the memory leaks in the new microservice, Disable the new microservice until the issue is resolved, Investigate the cause of the memory leaks in the new microservice]"


In [5]:
# 3. MAIN ACTION REQUEST using ai_gen (SQL syntax)
print("🎯 Step 3: Extracting main action request using ai_gen...")

if ai_functions_available:
    try:
        # Register DataFrame as temporary view for SQL access
        df_with_actions_parsed.createOrReplaceTempView("tickets_with_actions_parsed")
        
        # Use SQL to call ai_gen
        df_with_main_action = spark.sql("""
            SELECT
                *,
                ai_gen(
                    CONCAT(
                        'Extract the main action or request from this ticket description. Return only the specific action requested, not a list of items. If no clear action is requested, return None. Ticket description: ',
                        description
                    )
                ) AS action_requested
            FROM tickets_with_actions_parsed
        """)
        print("✅ Main action request extraction completed")
    except Exception as e:
        print(f"❌ Error in main action extraction: {e}")
        print("🔧 Using fallback approach...")
        ai_functions_available = False

if not ai_functions_available:
    # Rule-based main action extraction
    df_with_main_action = df_with_actions_parsed.withColumn(
        "action_requested",
        when(col("description").rlike("(?i)(install|setup|configure|deploy|update|fix|resolve|implement|create|build|test|monitor|backup|restore|migrate|upgrade|patch|secure|optimize|troubleshoot|investigate|analyze|review|document|train|schedule|plan|design|develop|maintain|manage|administer|operate|support|service)"),
             regexp_extract(col("description"), "(?i)(install|setup|configure|deploy|update|fix|resolve|implement|create|build|test|monitor|backup|restore|migrate|upgrade|patch|secure|optimize|troubleshoot|investigate|analyze|review|document|train|schedule|plan|design|develop|maintain|manage|administer|operate|support|service)", 1))
        .otherwise(lit("General request"))
    )
    print("✅ Rule-based main action extraction completed")

display(df_with_main_action.select("ticket_id", "short_description", "action_requested"))

# 4. TIMELINE EXTRACTION using ai_gen (SQL syntax)
print("⏰ Step 4: Extracting timeline information using ai_gen...")

if ai_functions_available:
    try:
        # Register DataFrame as temporary view for SQL access
        df_with_main_action.createOrReplaceTempView("tickets_with_main_action")
        
        # Use SQL to call ai_gen
        df_with_timeline = spark.sql("""
            SELECT
                *,
                ai_gen(
                    CONCAT(
                        'Extract timeline information from this ticket description. Look for due dates, deadlines, urgency indicators, or time references. Return the specific timeline mentioned, or None if no timeline is specified. Ticket description: ',
                        description
                    )
                ) AS timeline_extracted
            FROM tickets_with_main_action
        """)
        print("✅ Timeline extraction completed")
    except Exception as e:
        print(f"❌ Error in timeline extraction: {e}")
        print("🔧 Using fallback approach...")
        ai_functions_available = False

if not ai_functions_available:
    # Rule-based timeline extraction
    df_with_timeline = df_with_main_action.withColumn(
        "timeline_extracted",
        when(col("description").rlike("(?i)(urgent|asap|immediately|today|tomorrow|this week|next week|this month|deadline|due)"),
             regexp_extract(col("description"), "(?i)(urgent|asap|immediately|today|tomorrow|this week|next week|this month|deadline|due)", 1))
        .otherwise(lit("No specific timeline"))
    )
    print("✅ Rule-based timeline extraction completed")
display(df_with_timeline.select("ticket_id", "short_description", "timeline_extracted"))


🎯 Step 3: Extracting main action request using ai_gen...
✅ Main action request extraction completed


,ticket_id,short_description,action_requested
0,TICKET_001,Issue #1 - Critical,Investigate and resolve the issue with the website's slow performance.
1,TICKET_002,Issue #2 - Important,Fix the login system.
2,TICKET_003,Issue #3 - Help needed,Debug the API errors.
3,TICKET_004,Issue #4 - Important,Investigate the monitoring alerts to determine if they are false positives or indicative of an actual issue.
4,TICKET_005,Issue #5 - Urgent,Investigate the cause of the deployment failure.
5,TICKET_006,Issue #6 - Problem,"Investigate the missing customer data issue, potentially related to a data migration problem."
6,TICKET_007,Issue #7 - Critical,Fix the authentication issue caused by the new security patch.
7,TICKET_008,Issue #8 - Important,Find out what's causing the spike in cloud costs and optimize it.
8,TICKET_009,Issue #9 - Urgent,Fix the backup system.
9,TICKET_010,Issue #10 - Critical,Fix or disable the new microservice to prevent memory leaks and system crashes.


⏰ Step 4: Extracting timeline information using ai_gen...
✅ Timeline extraction completed


,ticket_id,short_description,timeline_extracted
0,TICKET_001,Issue #1 - Critical,"The timeline mentioned in the ticket description is: ""since this morning"". This indicates that the issue started at some point this morning, but no specific due date, deadline, or urgency indicator (like ""ASAP"" or ""urgent"") is mentioned beyond the fact that sales are being lost, implying a need for prompt attention."
1,TICKET_002,Issue #2 - Important,"The timeline information mentioned in the ticket description is:\n\n* ""ASAP"" (as soon as possible), indicating a high level of urgency\n* ""last week"", referencing a previous incident, but not providing a specific deadline or due date for the current issue.\n\nNo specific due date or deadline is mentioned, but the urgency is high."
2,TICKET_003,Issue #3 - Help needed,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The request is for general assistance with debugging an issue, but it does not include any time-sensitive information."
3,TICKET_004,Issue #4 - Important,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The description is a general request for help with an issue, but it does not include any time-related information."
4,TICKET_005,Issue #5 - Urgent,"The specific timeline mentioned is: ""this month"". This indicates that the issue has occurred three times within the current month, but it does not provide a specific due date or deadline."
5,TICKET_006,Issue #6 - Problem,"The timeline information mentioned in the ticket description is: ""immediately"". This indicates a sense of urgency, but does not provide a specific due date or deadline. There is also a reference to ""the last update"", which implies that the issue occurred recently, but the exact time frame is not specified. \n\nSo, the extracted timeline information is: ""immediately"" (indicating high urgency, but no specific date or time frame)."
6,TICKET_007,Issue #7 - Critical,"The timeline information mentioned in the ticket description is: ""before the security team gets involved"". This implies a sense of urgency, but does not specify a particular due date or deadline. However, it can be inferred that the issue needs to be resolved as soon as possible to avoid escalation to the security team. \n\nSince there is no specific date or time mentioned, the extracted timeline information is somewhat vague, but it does indicate a need for prompt action."
7,TICKET_008,Issue #8 - Important,"None \n\nThere is no specific due date, deadline, urgency indicator, or time reference mentioned in the ticket description, apart from the general mention of ""this month"", which is not a specific timeline for the task itself."
8,TICKET_009,Issue #9 - Urgent,"The timeline information mentioned in the ticket description is:\n\n* 3 days (the time since the last successful backup)\n\nThere is also an implied urgency to fix the issue as soon as possible to prevent potential problems, but no specific due date or deadline is mentioned."
9,TICKET_010,Issue #10 - Critical,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The description only mentions the need to fix or disable the microservice, but does not provide any time-related information."


In [6]:
# Action Items Categorization
print("🏷️ Step 5: Categorizing action items...")

# Explode action items to create one row per action item
df_action_items_exploded = df_with_timeline.select(
    "ticket_id",
    "short_description", 
    "action_requested",
    "priority_classified",
    "timeline_extracted",
    explode("action_items_extracted").alias("action_item")
).filter(col("action_item").isNotNull())

print(f"✅ Exploded {df_action_items_exploded.count()} individual action items")

# Categorize action items using ai_classify (SQL syntax)
if ai_functions_available:
    try:
        # Register DataFrame as temporary view for SQL access
        df_action_items_exploded.createOrReplaceTempView("action_items_exploded")
        
        # Use SQL to call ai_classify
        df_action_items_categorized = spark.sql("""
            SELECT
                *,
                ai_classify(action_item, ARRAY('Infrastructure', 'Security', 'Monitoring', 'Testing', 'Documentation', 'Network', 'Database', 'Application', 'Other')) as action_category
            FROM action_items_exploded
        """)
        print("✅ Action items categorization completed")
    except Exception as e:
        print(f"❌ Error in action categorization: {e}")
        print("🔧 Using fallback approach...")
        ai_functions_available = False

if not ai_functions_available:
    # Rule-based action categorization
    df_action_items_categorized = df_action_items_exploded.withColumn(
        "action_category",
        when(col("action_item").rlike("(?i)(server|infrastructure|cloud|vm|container|kubernetes|docker)"), lit("Infrastructure"))
        .when(col("action_item").rlike("(?i)(security|auth|permission|access|encrypt|firewall|vulnerability)"), lit("Security"))
        .when(col("action_item").rlike("(?i)(monitor|alert|log|metric|dashboard|health)"), lit("Monitoring"))
        .when(col("action_item").rlike("(?i)(test|testing|qa|validation|verify)"), lit("Testing"))
        .when(col("action_item").rlike("(?i)(document|doc|readme|guide|manual)"), lit("Documentation"))
        .when(col("action_item").rlike("(?i)(network|dns|load|balancer|proxy|vpn)"), lit("Network"))
        .when(col("action_item").rlike("(?i)(database|db|sql|data|table|query)"), lit("Database"))
        .when(col("action_item").rlike("(?i)(app|application|api|service|microservice)"), lit("Application"))
        .otherwise(lit("Other"))
    )
    print("✅ Rule-based action categorization completed")

display(df_action_items_categorized)


🏷️ Step 5: Categorizing action items...


✅ Exploded 26 individual action items
✅ Action items categorization completed


,ticket_id,short_description,action_requested,priority_classified,timeline_extracted,action_item,action_category
0,TICKET_001,Issue #1 - Critical,Investigate and resolve the issue with the website's slow performance.,Urgent Priority,"The timeline mentioned in the ticket description is: ""since this morning"". This indicates that the issue started at some point this morning, but no specific due date, deadline, or urgency indicator (like ""ASAP"" or ""urgent"") is mentioned beyond the fact that sales are being lost, implying a need for prompt attention.",Investigate the cause of the website's slow performance,Monitoring
1,TICKET_001,Issue #1 - Critical,Investigate and resolve the issue with the website's slow performance.,Urgent Priority,"The timeline mentioned in the ticket description is: ""since this morning"". This indicates that the issue started at some point this morning, but no specific due date, deadline, or urgency indicator (like ""ASAP"" or ""urgent"") is mentioned beyond the fact that sales are being lost, implying a need for prompt attention.",Check the database for potential issues,Database
2,TICKET_001,Issue #1 - Critical,Investigate and resolve the issue with the website's slow performance.,Urgent Priority,"The timeline mentioned in the ticket description is: ""since this morning"". This indicates that the issue started at some point this morning, but no specific due date, deadline, or urgency indicator (like ""ASAP"" or ""urgent"") is mentioned beyond the fact that sales are being lost, implying a need for prompt attention.",Look into the website's performance since this morning,Monitoring
3,TICKET_002,Issue #2 - Important,Fix the login system.,Urgent Priority,"The timeline information mentioned in the ticket description is:\n\n* ""ASAP"" (as soon as possible), indicating a high level of urgency\n* ""last week"", referencing a previous incident, but not providing a specific deadline or due date for the current issue.\n\nNo specific due date or deadline is mentioned, but the urgency is high.",Fix the login system,Application
4,TICKET_002,Issue #2 - Important,Fix the login system.,Urgent Priority,"The timeline information mentioned in the ticket description is:\n\n* ""ASAP"" (as soon as possible), indicating a high level of urgency\n* ""last week"", referencing a previous incident, but not providing a specific deadline or due date for the current issue.\n\nNo specific due date or deadline is mentioned, but the urgency is high.",Investigate the cause of the login system failure,Application
5,TICKET_002,Issue #2 - Important,Fix the login system.,Urgent Priority,"The timeline information mentioned in the ticket description is:\n\n* ""ASAP"" (as soon as possible), indicating a high level of urgency\n* ""last week"", referencing a previous incident, but not providing a specific deadline or due date for the current issue.\n\nNo specific due date or deadline is mentioned, but the urgency is high.",Prevent the login system from breaking again,Security
6,TICKET_003,Issue #3 - Help needed,Debug the API errors.,Medium Priority,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The request is for general assistance with debugging an issue, but it does not include any time-sensitive information.",Debug the API to identify the cause of the random errors,Testing
7,TICKET_004,Issue #4 - Important,Investigate the monitoring alerts to determine if they are false positives or indicative of an actual issue.,High Priority,"None \n\nThere is no specific timeline mentioned in the ticket description, such as due dates, deadlines, or urgency indicators. The description is a general request for help with an issue, but it does not include any time-related information.",Investigate monitoring alerts,Monitoring
8,TICKET_004,Issue #4 - Important,Investigate the monitoring alerts to determine if they are false positives or indicative of an actual issue.,High Priorit

In [7]:
# Save Extracted Data
print("💾 Step 6: Saving extracted data to Unity Catalog...")

try:
    # Cache the DataFrames to avoid recomputation
    df_with_timeline.cache()
    df_action_items_categorized.cache()
    
    # Save the main extracted data to Unity Catalog
    df_with_timeline.write.format("delta").mode("overwrite").saveAsTable(TABLES["tickets_with_actions"])
    print("✅ Main data saved successfully!")
    
    # Save the categorized action items to Unity Catalog  
    df_action_items_categorized.write.format("delta").mode("overwrite").saveAsTable(TABLES["action_items_detailed"])
    print("✅ Action items saved successfully!")
    
    print("✅ AI-extracted data saved to Unity Catalog Delta tables:")
    print(f"📊 Main data: {TABLES['tickets_with_actions']}")
    print(f"📋 Action items: {TABLES['action_items_detailed']}")
    
    # Show summary statistics (with error handling)
    print(f"\n📈 Summary Statistics:")
    try:
        ticket_count = df_with_timeline.count()
        print(f"   Total tickets processed: {ticket_count}")
    except Exception as e:
        print(f"   Error counting tickets: {e}")
    
    try:
        action_count = df_action_items_categorized.count()
        print(f"   Total action items extracted: {action_count}")
    except Exception as e:
        print(f"   Error counting action items: {e}")
    
    # Show action category distribution (with error handling)
    print(f"\n📊 Action Category Distribution:")
    try:
        df_action_items_categorized.groupBy("action_category").count().orderBy(desc("count")).show()
    except Exception as e:
        print(f"   Error showing action categories: {e}")
    
    # Show priority distribution (with error handling)
    print(f"\n🎯 Priority Distribution:")
    try:
        df_with_timeline.groupBy("priority_classified").count().orderBy(desc("count")).show()
    except Exception as e:
        print(f"   Error showing priority distribution: {e}")
    
    print("\n🎯 Ready for next notebook: 03_ai_classification.ipynb")
    
except Exception as e:
    print(f"❌ Error saving data: {e}")
    print("🔍 Trying to save to temporary tables...")
    try:
        df_with_timeline.write.format("delta").mode("overwrite").saveAsTable("tickets_with_actions_temp")
        df_action_items_categorized.write.format("delta").mode("overwrite").saveAsTable("action_items_detailed_temp")
        print("✅ Saved to temporary tables")
    except Exception as e2:
        print(f"❌ Error with temp tables: {e2}")

# Alternative: Simple completion message without expensive operations
print("\n" + "="*50)
print("🎉 NOTEBOOK COMPLETED SUCCESSFULLY!")
print("="*50)
print("✅ Data has been saved to Unity Catalog")
print("✅ Action items have been extracted and categorized")
print("✅ Ready to proceed to next notebook: 03_ai_classification.ipynb")
print("="*50)


💾 Step 6: Saving extracted data to Unity Catalog...


✅ Main data saved successfully!


✅ Action items saved successfully!
✅ AI-extracted data saved to Unity Catalog Delta tables:
📊 Main data: quickstart_catalog_vkm_external.classify_tickets.tickets_with_actions
📋 Action items: quickstart_catalog_vkm_external.classify_tickets.action_items_detailed

📈 Summary Statistics:


   Total tickets processed: 10


   Total action items extracted: 26

📊 Action Category Distribution:


+---------------+-----+
|action_category|count|
+---------------+-----+
|     Monitoring|    7|
|    Application|    6|
|        Testing|    4|
|       Security|    3|
| Infrastructure|    3|
|       Database|    3|
+---------------+-----+


🎯 Priority Distribution:


+-------------------+-----+
|priority_classified|count|
+-------------------+-----+
|    Urgent Priority|    6|
|      High Priority|    3|
|    Medium Priority|    1|
+-------------------+-----+


🎯 Ready for next notebook: 03_ai_classification.ipynb

🎉 NOTEBOOK COMPLETED SUCCESSFULLY!
✅ Data has been saved to Unity Catalog
✅ Action items have been extracted and categorized
✅ Ready to proceed to next notebook: 03_ai_classification.ipynb
